In [2]:
# 1
# Разовая конвертация CSV → Parquet
#
# Цель:
#   Конвертировать большой CSV-файл датасета LycoS-Unicas-IDS2018
#   в более компактный и удобный для дальнейшей обработки формат Parquet.
#
# Почему это нужно:
#   - исходный CSV занимает около 5.5 GB;
#   - при чтении через pandas CSV может потреблять 7–8 GB RAM;
#   - Parquet хранит типизированные данные, быстрее читается и занимает меньше места;
#   - потоковая обработка через pyarrow позволяет не загружать весь CSV в память.
#
# После успешной конвертации дальнейшие этапы пайплайна должны использовать Parquet.


import pandas as pd
import pyarrow as pa
import pyarrow.csv as pc
import pyarrow.parquet as pq

from pathlib import Path


CSV_PATH     = Path("../data/raw/LycoS-Unicas-IDS2018.csv")
PARQUET_PATH = Path("../data/raw/LycoS-Unicas-IDS2018.parquet")


if not PARQUET_PATH.exists():
    print("Конвертация CSV → Parquet")

    # Читаем только заголовок CSV, чтобы получить список колонок.
    col_names = pd.read_csv(CSV_PATH, nrows=0).columns.tolist()

    # В датасете предполагается, что последняя колонка — целевая переменная.
    feature_cols = col_names[:-1]

    # Признаки приводятся к float32 для уменьшения размера данных.
    # Целевая переменная не изменяется и будет определена pyarrow автоматически.
    convert_options = pc.ConvertOptions(
        column_types={col: pa.float32() for col in feature_cols}
    )

    writer = None

    with pc.open_csv(CSV_PATH, convert_options=convert_options) as reader:
        for batch in reader:  # каждый batch ≈ 64 MB из CSV
            if writer is None:
                writer = pq.ParquetWriter(
                    PARQUET_PATH,
                    batch.schema,
                    compression="snappy",
                )
            writer.write_batch(batch)
    writer.close()

    size_mb = Path(PARQUET_PATH).stat().st_size / 1024**2
    print(f"Готово. Размер Parquet: {size_mb:.0f} MB")
else:
    print(f"Parquet уже есть: {PARQUET_PATH}")

Конвертация CSV → Parquet
Готово. Размер Parquet: 1513 MB


In [3]:
# 2
"""
Настройка DuckDB для работы с большими табличными данными.

В этой ячейке создаётся соединение DuckDB и задаются ограничения на использование
оперативной памяти. Также настраивается директория для временных файлов DuckDB,
которые могут использоваться при выполнении larger-than-memory операций.

Дополнительно определена функция мягкой очистки памяти после выполнения ячеек
в Jupyter/IPython. Эта функция не гарантирует полного сброса внутреннего кэша DuckDB,
но может помочь снизить удержание памяти между тяжёлыми вычислительными шагами.
"""

import gc
from pathlib import Path

import duckdb



DUCKDB_MEMORY_LIMIT = "4GB"
DUCKDB_TEMP_DIR = Path("/tmp/duckdb_spill")
AUTO_CLEANUP_AFTER_CELL = False


# Создаём директорию для временных файлов DuckDB.
DUCKDB_TEMP_DIR.mkdir(parents=True, exist_ok=True)


# Создаём соединение DuckDB.
# Без указания пути используется in-memory соединение.
con = duckdb.connect()

# Ограничиваем память, используемую DuckDB.
# Важно: memory_limit не является строгим пределом RSS-памяти всего Python-процесса.
con.execute(f"SET memory_limit = '{DUCKDB_MEMORY_LIMIT}'")

# Директория для временных файлов при spilling на диск.
con.execute(f"SET temp_directory = '{DUCKDB_TEMP_DIR}'")


def cleanup_duckdb_memory():
    """
    Выполняет мягкую очистку памяти после тяжёлых операций DuckDB.

    Важно:
        Эта функция не является гарантированным сбросом всего внутреннего кэша DuckDB.
        Она временно снижает memory_limit, затем возвращает рабочее значение,
        после чего запускает сборщик мусора Python.
    """
    try:
        con.execute("SET memory_limit = '128MB'")
        con.execute(f"SET memory_limit = '{DUCKDB_MEMORY_LIMIT}'")
        gc.collect()
    except Exception as exc:
        print(f"[DuckDB cleanup warning] {exc}")


def register_duckdb_cleanup_callback():
    """
    Регистрирует cleanup_duckdb_memory как post_execute callback в IPython/Jupyter.

    Используется только в интерактивной среде. Перед регистрацией удаляются ранее
    зарегистрированные callback-функции с таким же именем, чтобы избежать дублей.
    """
    ip = get_ipython() if "get_ipython" in globals() else None

    if ip is None:
        print("IPython-среда не обнаружена: callback не зарегистрирован.")
        return

    for fn in list(ip.events.callbacks.get("post_execute", [])):
        if getattr(fn, "__name__", "") == "cleanup_duckdb_memory":
            ip.events.unregister("post_execute", fn)

    ip.events.register("post_execute", cleanup_duckdb_memory)
    print("Автоматическая мягкая очистка DuckDB после каждой ячейки включена.")


if AUTO_CLEANUP_AFTER_CELL:
    register_duckdb_cleanup_callback()


print(f"DuckDB готов: memory_limit = {DUCKDB_MEMORY_LIMIT}, temp_directory = {DUCKDB_TEMP_DIR}")

DuckDB готов: memory_limit = 4GB, temp_directory = /tmp/duckdb_spill


In [4]:
# Диагностика датасета для проектирования препроцессинга (read-only)
import duckdb
import pandas as pd
from pathlib import Path

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")


PARQUET_PATH = Path("../data/raw/LycoS-Unicas-IDS2018.parquet")

try:
    con
except NameError:
    con = duckdb.connect()

p = Path(PARQUET_PATH).as_posix()

# 1. Схема + статистики по всем колонкам одним проходом.
summary = con.execute(f"SUMMARIZE SELECT * FROM read_parquet('{p}')").df()

n_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{p}')").fetchone()[0]

print("=" * 90)
print(f"1. SCHEMA + СТАТИСТИКИ   |   строк: {n_rows:,}   колонок: {len(summary)}")
print("=" * 90)
show_cols = ["column_name", "column_type", "min", "max", "approx_unique", "avg", "std", "null_percentage"]
print(summary[show_cols].to_string(index=False))

# 2. Сводка по типам колонок.
print("\n" + "=" * 90)
print("2. ТИПЫ КОЛОНОК")
print("=" * 90)
print(summary["column_type"].value_counts().to_string())

# 3. ip_prot — точная кардинальность и распределение (для one-hot).
print("\n" + "=" * 90)
print("3. ip_prot (точное распределение)")
print("=" * 90)
ipp = con.execute(f"""
    SELECT CAST(ip_prot AS INTEGER) AS ip_prot,
           COUNT(*) AS n,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 3) AS pct
    FROM read_parquet('{p}')
    GROUP BY 1
    ORDER BY n DESC
""").df()
print(ipp.to_string(index=False))

# 4. label — точные строковые значения (проверка эталонного 'Benign').
print("\n" + "=" * 90)
print("4. label (точные значения)")
print("=" * 90)
lab = con.execute(f"""
    SELECT label, COUNT(*) AS n
    FROM read_parquet('{p}')
    GROUP BY 1
    ORDER BY n DESC
""").df()
print(lab.to_string(index=False))

# 5. Риск float32: целочисленные счётчики, которые могут терять точность (> 2^24).
print("\n" + "=" * 90)
print("5. РИСК float32  (max > 16,777,216)")
print("=" * 90)
num = summary.copy()
num["max_num"] = pd.to_numeric(num["max"], errors="coerce")
num["min_num"] = pd.to_numeric(num["min"], errors="coerce")
risk = num[num["max_num"] > 16_777_216][["column_name", "column_type", "min", "max"]]
print(risk.to_string(index=False) if not risk.empty else "Колонок с риском не найдено ✓")

# 6. Низкокардинальные признаки — кандидаты в бинарные / категориальные / константы.
print("\n" + "=" * 90)
print("6. НИЗКОКАРДИНАЛЬНЫЕ признаки (approx_unique <= 5)")
print("=" * 90)
lowcard = summary[summary["approx_unique"] <= 5][["column_name", "column_type", "min", "max", "approx_unique"]]
print(lowcard.to_string(index=False) if not lowcard.empty else "Нет")

# 7. Признаки с отрицательными значениями (ожидаем только TCP-window c -1).
print("\n" + "=" * 90)
print("7. Признаки с отрицательным min")
print("=" * 90)
negs = num[num["min_num"] < 0][["column_name", "column_type", "min", "max"]]
print(negs.to_string(index=False) if not negs.empty else "Отрицательных нет")

# 8. Несколько примеров строк (транспонированно, чтобы видеть формат значений).
print("\n" + "=" * 90)
print("8. ПРИМЕРЫ СТРОК")
print("=" * 90)
sample = con.execute(f"SELECT * FROM read_parquet('{p}') USING SAMPLE 5 ROWS").df()
print(sample.T.to_string())

1. SCHEMA + СТАТИСТИКИ   |   строк: 13,691,268   колонок: 78
           column_name column_type    min              max  approx_unique                 avg                 std  null_percentage
              dst_port       FLOAT    0.0          65535.0          28759  1048.7526352562816  3578.8635242929836           0.0000
               ip_prot       FLOAT    1.0             17.0              4   9.170069638546261    4.99855039967288           0.0000
         flow_duration       FLOAT    0.0      120000000.0        3878766   9798519.276794013  28956142.015817832           0.0000
         down_up_ratio       FLOAT    0.0            24.25          21303  0.9649331831053447  0.4271243536166334           0.0000
           pkt_len_max       FLOAT    0.0          65496.0           1988  1961.5993216260174   9387.167843821246           0.0000
           pkt_len_min       FLOAT    0.0           1472.0            345  13.712862460949562  40.119486730573826           0.0000
          pkt_len_mean

In [5]:
# 3
"""
Первичный анализ датасета: размер таблицы и распределение классов.

Ячейка:
    1. Определяет количество строк в Parquet-файле.
    2. Определяет количество колонок.
    3. Рассчитывает распределение целевой переменной label:
       - абсолютное количество объектов каждого класса;
       - долю класса в процентах.

Этот шаг используется для проверки корректности загрузки данных и анализа
дисбаланса классов перед обучением моделей.
"""

from pathlib import Path

import duckdb


# Если соединение DuckDB уже было создано на предыдущем шаге,
# лучше переиспользовать его, чтобы не потерять настройки memory_limit и temp_directory.
try:
    con
except NameError:
    con = duckdb.connect()
    con.execute("SET memory_limit = '4GB'")
    con.execute("SET temp_directory = '/tmp/duckdb_spill'")


parquet_path = Path(PARQUET_PATH)

if not parquet_path.exists():
    raise FileNotFoundError(f"Parquet-файл не найден: {parquet_path}")


# Количество строк.
# Для Parquet DuckDB может использовать метаданные row groups,
# поэтому COUNT(*) обычно выполняется без полного чтения всех данных.
n_rows = con.execute(f"""
    SELECT COUNT(*) AS n_rows
    FROM read_parquet('{parquet_path.as_posix()}')
""").fetchone()[0]


# Количество колонок.
# LIMIT 0 возвращает только схему таблицы без загрузки строк.
n_cols = len(con.execute(f"""
    SELECT *
    FROM read_parquet('{parquet_path.as_posix()}')
    LIMIT 0
""").df().columns)


print(f"Датасет: {n_rows:,} строк × {n_cols} колонок")


# Распределение классов целевой переменной.
dist = con.execute(f"""
    SELECT
        label,
        COUNT(*) AS count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM read_parquet('{parquet_path.as_posix()}')
    GROUP BY label
    ORDER BY count DESC
""").df()


print("Распределение классов:")
print(dist.to_string(index=False))

Датасет: 13,691,268 строк × 78 колонок
Распределение классов:
                     label    count     pct
                    Benign 10000000 73.0400
                  DoS Hulk  1802966 13.1700
                 DDoS HOIC  1074379  7.8500
            DDoS LOIC-HTTP   289328  2.1100
               FTP-Patator   190300  1.3900
          DoS Slowhttptest   105550  0.7700
                       Bot    96154  0.7000
               SSH-Patator    92648  0.6800
             DoS GoldenEye    26861  0.2000
             DoS Slowloris    10274  0.0800
             DDoS LOIC-UDP     2382  0.0200
  Web Attack - Brute Force      260  0.0000
          Web Attack - XSS      116  0.0000
Web Attack - Sql Injection       50  0.0000


In [6]:
# 4
"""
Инициализация списков колонок и SQL hash-выражений для аудита датасета.

Эта ячейка выполняется один раз перед аудитом данных. Полученные переменные
используются в следующих проверках:
    1. поиск полных дубликатов строк;
    2. поиск потенциальных конфликтов разметки;
    3. анализ совпадающих признаковых описаний.

Определения:
    all_cols      — все колонки датасета;
    feature_cols  — все признаки, кроме целевой переменной label;
    feat_no_dst   — признаки без label и dst_port.

Важно:
    Исключение dst_port используется только для отдельной проверки, где нужно
    сравнить объекты без учёта порта назначения. Для строгого поиска конфликтов
    разметки следует использовать все признаки без label.
"""

from pathlib import Path


LABEL_COL = "label"
DST_PORT_COL = "dst_port"

parquet_path = Path(PARQUET_PATH)

if not parquet_path.exists():
    raise FileNotFoundError(f"Parquet-файл не найден: {parquet_path}")


def quote_identifier(column_name: str) -> str:
    """
    Экранирует имя колонки для безопасной вставки в SQL-запрос DuckDB.

    Двойные кавычки внутри имени колонки удваиваются по SQL-правилам.
    """
    return '"' + column_name.replace('"', '""') + '"'


# Получаем список колонок без чтения строк датасета.
all_cols = con.execute(f"""
    SELECT *
    FROM read_parquet('{parquet_path.as_posix()}')
    LIMIT 0
""").df().columns.tolist()


if LABEL_COL not in all_cols:
    raise ValueError(f"В датасете не найдена целевая колонка: {LABEL_COL}")


feature_cols = [col for col in all_cols if col != LABEL_COL]

if DST_PORT_COL in feature_cols:
    feat_no_dst = [col for col in feature_cols if col != DST_PORT_COL]
else:
    feat_no_dst = feature_cols.copy()
    print(f"Предупреждение: колонка {DST_PORT_COL!r} не найдена, список feat_no_dst совпадает с feature_cols.")


# Хэш по всем колонкам используется для поиска полных дубликатов строк.
all_hash = "hash(" + ", ".join(quote_identifier(col) for col in all_cols) + ")"

# Хэш по всем признакам без label используется для строгого поиска конфликтов разметки.
feature_hash = "hash(" + ", ".join(quote_identifier(col) for col in feature_cols) + ")"

# Хэш по признакам без label и dst_port используется для дополнительного анализа
# совпадений без учёта порта назначения.
feature_no_dst_hash = "hash(" + ", ".join(quote_identifier(col) for col in feat_no_dst) + ")"


print(f"Всего колонок: {len(all_cols)}")
print(f"Признаков: {len(feature_cols)} (исключили {LABEL_COL})")
print(f"Признаков: {len(feat_no_dst)} (исключили {LABEL_COL} и {DST_PORT_COL})")

Всего колонок: 78
Признаков: 77 (исключили label)
Признаков: 76 (исключили label и dst_port)


In [7]:
# 5
"""
Аудит признаков на наличие некорректных числовых значений: NULL, NaN и Infinity.

Цель проверки:
    Выявить значения, которые могут привести к ошибкам на этапе preprocessing
    или обучения моделей машинного обучения.

Проверяемые случаи:
    1. NULL — отсутствующее значение на уровне таблицы/Parquet.
    2. NaN — специальное float-значение IEEE-754.
    3. ±Infinity — положительная или отрицательная бесконечность.

Проверка выполняется средствами DuckDB напрямую по Parquet-файлу без загрузки
всего датасета в память.
"""

from pathlib import Path

import pandas as pd


parquet_path = Path(PARQUET_PATH)

if not parquet_path.exists():
    raise FileNotFoundError(f"Parquet-файл не найден: {parquet_path}")


def quote_identifier(column_name: str) -> str:
    """
    Экранирует имя колонки для безопасной вставки в SQL-запрос DuckDB.
    """
    return '"' + column_name.replace('"', '""') + '"'


def count_bad_values(predicate_template: str, check_name: str, columns: list[str]) -> pd.Series:
    """
    Считает количество строк с некорректными значениями по каждой колонке.

    Parameters
    ----------
    predicate_template:
        SQL-шаблон условия с плейсхолдером {col}.
        Например: '{col} IS NULL OR isnan({col})'.

    check_name:
        Название проверки для вывода в лог.

    columns:
        Список колонок, по которым выполняется проверка.

    Returns
    -------
    pd.Series
        Series вида:
            index   — имя колонки;
            value   — количество строк, удовлетворяющих условию.
        Возвращаются только колонки, где количество проблемных строк больше нуля.
    """
    if not columns:
        print(f"{check_name}: список колонок пуст.")
        return pd.Series(dtype="int64")

    expressions = []

    for col in columns:
        col_sql = quote_identifier(col)

        expressions.append(
            f"""
            SUM(
                CASE
                    WHEN {predicate_template.format(col=col_sql)}
                    THEN 1
                    ELSE 0
                END
            ) AS {col_sql}
            """
        )

    query = f"""
        SELECT
            {", ".join(expressions)}
        FROM read_parquet('{parquet_path.as_posix()}')
    """

    row = con.execute(query).df().iloc[0]
    found = row[row > 0].astype("int64")

    if found.empty:
        print(f"{check_name}: не найдено ✓")
    else:
        print(f"{check_name}: найдено в {len(found)} колонках:")
        print(found.to_string())

    return found


# NULL и NaN — разные сущности:
#   - NULL хранится как отсутствующее значение;
#   - NaN является специальным значением внутри float-колонки.
#
# isnan(NULL) не даёт TRUE, поэтому NULL и NaN проверяются совместно.
nan_null_counts = count_bad_values(
    predicate_template="{col} IS NULL OR isnan({col})",
    check_name="NaN / NULL",
    columns=feature_cols,
)


# isinf() выявляет как +Infinity, так и -Infinity.
inf_counts = count_bad_values(
    predicate_template="isinf({col})",
    check_name="Infinity",
    columns=feature_cols,
)

NaN / NULL: не найдено ✓
Infinity: не найдено ✓


In [8]:
# 6
"""
Аудит константных признаков.

Цель:
    Найти признаки, которые принимают одно и то же значение во всех строках
    текущего датасета.

Важно:
    Константные признаки не несут обучающего сигнала внутри текущего датасета,
    так как их дисперсия равна нулю. Однако в cross-dataset сценарии такие
    признаки не удаляются физически из Parquet-файла, чтобы сохранить единую
    схему признакового пространства между обучающим и внешним тестовым датасетом.

Результат:
    Формируется список const_found с именами константных признаков.
    Исходный Parquet-файл не изменяется.
"""

import pandas as pd


CONST_DESCRIPTIONS = {
    "flag_urg": "суммарное количество TCP-пакетов с URG-флагом в потоке",
    "fwd_flag_urg": "количество URG-пакетов в прямом направлении клиент → сервер",
    "bwd_flag_urg": "количество URG-пакетов в обратном направлении сервер → клиент",
}


def quote_identifier(column_name: str) -> str:
    """
    Экранирует имя колонки для безопасной вставки в SQL-запрос DuckDB.
    """
    return '"' + column_name.replace('"', '""') + '"'


# Для каждого признака проверяем равенство MIN и MAX.
# Если MIN(feature) = MAX(feature), значит признак константен
# на всём текущем датасете.
const_exprs = ", ".join(
    [
        f"MIN({quote_identifier(col)}) = MAX({quote_identifier(col)}) AS {quote_identifier(col)}"
        for col in feature_cols
    ]
)

const_row = con.execute(f"""
    SELECT
        {const_exprs}
    FROM read_parquet('{PARQUET_PATH}')
""").df().iloc[0]


const_found = list(const_row[const_row == True].index)


if not const_found:
    print("Константные признаки не обнаружены ✓")

else:
    print(f"Найдено константных признаков: {len(const_found)}\n")

    const_report_rows = []

    for col in const_found:
        col_sql = quote_identifier(col)

        value = con.execute(f"""
            SELECT {col_sql}
            FROM read_parquet('{PARQUET_PATH}')
            LIMIT 1
        """).fetchone()[0]

        description = CONST_DESCRIPTIONS.get(col, "описание не задано")

        const_report_rows.append(
            {
                "feature": col,
                "constant_value": value,
                "description": description,
            }
        )

    const_report = pd.DataFrame(const_report_rows)

    print(const_report.to_string(index=False))

    print("""
Интерпретация:
  Константные признаки имеют нулевую дисперсию в текущем датасете.
  Следовательно, при обучении на этом датасете модель не сможет получить
  из них полезный разделяющий сигнал.

  Однако признаки не удаляются из Parquet-файла, поскольку в cross-dataset
  сценарии важно сохранить единую схему признакового пространства. Внешний
  тестовый датасет должен подаваться в модель с тем же набором колонок,
  что и обучающий датасет.
""")

Найдено константных признаков: 3

     feature  constant_value                                                   description
    flag_urg          0.0000        суммарное количество TCP-пакетов с URG-флагом в потоке
fwd_flag_urg          0.0000   количество URG-пакетов в прямом направлении клиент → сервер
bwd_flag_urg          0.0000 количество URG-пакетов в обратном направлении сервер → клиент

Интерпретация:
  Константные признаки имеют нулевую дисперсию в текущем датасете.
  Следовательно, при обучении на этом датасете модель не сможет получить
  из них полезный разделяющий сигнал.

  Однако признаки не удаляются из Parquet-файла, поскольку в cross-dataset
  сценарии важно сохранить единую схему признакового пространства. Внешний
  тестовый датасет должен подаваться в модель с тем же набором колонок,
  что и обучающий датасет.



In [9]:
# 7
# Аудит отрицательных значений (read-only, без записи).
#
# Инвариант raw-датасета:
#   Отрицательные значения допустимы ТОЛЬКО в TCP-window признаках
#   (fwd/bwd_tcp_init_win_bytes) и ТОЛЬКО как sentinel -1
#   ("обратное окно отсутствует / поток без ответной стороны").
#
# Здесь мы НИЧЕГО не заменяем. Sentinel -1 сохраняется в raw и processed.
# Решение об обработке -1 принимается в слое препроцессинга (ноутбук 2),
# отдельно по модели: деревья едят -1 как есть; для linear/nn-ветки
# добавляется indicator bwd_tcp_init_win_absent, а -1 -> 0 перед скейлингом.

from pathlib import Path

EXPECTED_NEGATIVE_COLS = ["fwd_tcp_init_win_bytes", "bwd_tcp_init_win_bytes"]
parquet_path = Path(PARQUET_PATH)

def quote_identifier(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'
def sql_path(path: Path) -> str:
    return str(path).replace("'", "''")

neg_exprs = ", ".join(
    f"SUM(CASE WHEN {quote_identifier(c)} < 0 THEN 1 ELSE 0 END) AS {quote_identifier(c)}"
    for c in feature_cols
)
neg_row = con.execute(f"""
    SELECT {neg_exprs} FROM read_parquet('{sql_path(parquet_path)}')
""").df().iloc[0]
neg_found = neg_row[neg_row > 0].sort_values(ascending=False)

print("Аудит отрицательных значений")
if neg_found.empty:
    print("Отрицательные значения не обнаружены.")
else:
    for col, cnt in neg_found.items():
        print(f"  {col:<35} {int(cnt):>12,} строк")

    # Инвариант 1: негативы только в ожидаемых TCP-window колонках.
    unexpected = sorted(set(neg_found.index) - set(EXPECTED_NEGATIVE_COLS))
    assert not unexpected, f"Отрицательные значения в неожиданных признаках: {unexpected}"

    # Инвариант 2: в этих колонках отрицательное значение только -1.
    for col in [c for c in EXPECTED_NEGATIVE_COLS if c in neg_found.index]:
        bad = con.execute(f"""
            SELECT COUNT(*) FROM read_parquet('{sql_path(parquet_path)}')
            WHERE {quote_identifier(col)} < 0 AND {quote_identifier(col)} != -1
        """).fetchone()[0]
        assert bad == 0, f"{col}: найдено отрицательное != -1 ({bad} строк)"

    print("\nИнвариант соблюдён: отрицательные значения только в TCP-window и только -1 ✓")
    print("Sentinel -1 сохраняется как есть (обработка — в препроцессинге обучения).")

Аудит отрицательных значений
  bwd_tcp_init_win_bytes                 4,776,157 строк
  fwd_tcp_init_win_bytes                 3,990,032 строк

Инвариант соблюдён: отрицательные значения только в TCP-window и только -1 ✓
Sentinel -1 сохраняется как есть (обработка — в препроцессинге обучения).


In [10]:
# 8
"""
Анализ распределения IP-протоколов в датасете.

Цель:
    Проверить, какие сетевые протоколы представлены в данных, и оценить
    распределение benign/malicious трафика внутри каждого протокола.

Зачем это нужно:
    ip_prot является внешней наблюдаемой характеристикой потока и отражает
    тип сетевого взаимодействия: TCP, UDP, ICMP и т.д. В отличие от dst_port,
    этот признак не идентифицирует конкретный сервис, а задаёт общий класс
    протокольного поведения.

Важно:
    В бинарной постановке все метки, отличные от Benign, агрегируются
    в класс malicious.
"""

import pandas as pd


PROTO_NAMES = {
    0: "HOPOPT",
    1: "ICMP",
    2: "IGMP",
    6: "TCP",
    17: "UDP",
    41: "IPv6",
    47: "GRE",
    50: "ESP",
    58: "ICMPv6",
    89: "OSPF",
    132: "SCTP",
}


proto_dist = con.execute(f"""
    WITH proto_counts AS (
        SELECT
            CAST(ip_prot AS INTEGER) AS proto,
            COUNT(*) AS count,
            SUM(CASE WHEN label = 'Benign' THEN 1 ELSE 0 END) AS benign,
            SUM(CASE WHEN label != 'Benign' THEN 1 ELSE 0 END) AS malicious
        FROM read_parquet('{PARQUET_PATH}')
        GROUP BY CAST(ip_prot AS INTEGER)
    )
    SELECT
        proto,
        count,
        ROUND(count * 100.0 / SUM(count) OVER (), 2) AS pct,
        benign,
        malicious
    FROM proto_counts
    ORDER BY count DESC
""").df()


proto_dist["name"] = proto_dist["proto"].map(PROTO_NAMES).fillna("UNKNOWN")

proto_dist = proto_dist[
    ["proto", "name", "count", "pct", "benign", "malicious"]
]


print("Распределение IP-протоколов:")
print(proto_dist.to_string(index=False))

Распределение IP-протоколов:
 proto name   count     pct       benign    malicious
     6  TCP 9701236 70.8600 6012350.0000 3688886.0000
    17  UDP 3958719 28.9100 3956360.0000    2359.0000
     1 ICMP   18384  0.1300   18361.0000      23.0000
     2 IGMP   12929  0.0900   12929.0000       0.0000


In [11]:
# 9 Сборка processed-датасета из НЕИЗМЕНЯЕМОГО raw.
#
#   * raw (data/raw/...parquet) не мутируется никогда;
#   * исключаем dst_port (сервисный идентификатор, 28 759 уникальных);
#   * добавляем label_binary (Benign -> 0, иначе -> 1), label сохраняем;
#   * -1 в TCP-window СОХРАНЯЕМ как sentinel (его обработка — в препроцессинге обучения);
#   * константы flag_urg/* оставляем ради паритета схемы с cross-dataset LycoS17.

import shutil
from pathlib import Path

RAW_PARQUET_PATH   = Path(PARQUET_PATH)
CLEAN_PARQUET_PATH = Path("../data/processed/dataset.parquet")
CLEAN_PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)

LABEL_COL, BINARY_LABEL_COL, BENIGN_LABEL = "label", "label_binary", "Benign"
DROP_COLS = {"dst_port"}

def quote_identifier(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'
def sql_path(path: Path) -> str:
    return str(path).replace("'", "''")

if not RAW_PARQUET_PATH.exists():
    raise FileNotFoundError(f"raw не найден: {RAW_PARQUET_PATH}")

raw_cols = con.execute(
    f"SELECT * FROM read_parquet('{sql_path(RAW_PARQUET_PATH)}') LIMIT 0"
).df().columns.tolist()

# Источник обязан быть чистым raw.
assert LABEL_COL in raw_cols, f"в raw нет {LABEL_COL!r}"
assert BINARY_LABEL_COL not in raw_cols, "raw мутирован: в нём уже есть label_binary"
raw_neg = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{sql_path(RAW_PARQUET_PATH)}')
    WHERE "fwd_tcp_init_win_bytes" < 0 OR "bwd_tcp_init_win_bytes" < 0
""").fetchone()[0]
assert raw_neg > 0, "raw мутирован: sentinel -1 уже затёрт"
raw_n = con.execute(f"SELECT COUNT(*) FROM read_parquet('{sql_path(RAW_PARQUET_PATH)}')").fetchone()[0]

keep_cols = [c for c in raw_cols if c not in DROP_COLS]
if DROP_COLS - set(raw_cols):
    print(f"Предупреждение: нет колонок для дропа: {sorted(DROP_COLS - set(raw_cols))}")

select_parts = [quote_identifier(c) for c in keep_cols]
select_parts.append(
    f"CASE WHEN {quote_identifier(LABEL_COL)} = '{BENIGN_LABEL}' THEN 0 ELSE 1 END "
    f"AS {quote_identifier(BINARY_LABEL_COL)}"
)

tmp_path = CLEAN_PARQUET_PATH.with_suffix(".tmp.parquet")
if tmp_path.exists():
    tmp_path.unlink()

con.execute(f"""
    COPY (
        SELECT {", ".join(select_parts)}
        FROM read_parquet('{sql_path(RAW_PARQUET_PATH)}')
    )
    TO '{sql_path(tmp_path)}' (FORMAT PARQUET, COMPRESSION SNAPPY)
""")

# Verify до атомарной замены.
new_cols = con.execute(f"SELECT * FROM read_parquet('{sql_path(tmp_path)}') LIMIT 0").df().columns.tolist()
assert "dst_port" not in new_cols, "dst_port не исключён"
assert BINARY_LABEL_COL in new_cols, "label_binary не создан"
new_n = con.execute(f"SELECT COUNT(*) FROM read_parquet('{sql_path(tmp_path)}')").fetchone()[0]
assert new_n == raw_n, f"строк изменилось: raw={raw_n}, processed={new_n}"
new_neg = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{sql_path(tmp_path)}')
    WHERE "fwd_tcp_init_win_bytes" < 0 OR "bwd_tcp_init_win_bytes" < 0
""").fetchone()[0]
assert new_neg == raw_neg, "sentinel -1 не сохранён"
mism = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{sql_path(tmp_path)}')
    WHERE (CASE WHEN {quote_identifier(LABEL_COL)} = '{BENIGN_LABEL}' THEN 0 ELSE 1 END)
          != {quote_identifier(BINARY_LABEL_COL)}
""").fetchone()[0]
assert mism == 0, f"label_binary не соответствует label: {mism}"

shutil.move(str(tmp_path), str(CLEAN_PARQUET_PATH))

n_feat = len(keep_cols) - 1  # минус label
size_mb = CLEAN_PARQUET_PATH.stat().st_size / 1024**2
print(f"processed: {CLEAN_PARQUET_PATH}  ({size_mb:.0f} MB)")
print(f"строк: {new_n:,}  |  колонок: {len(new_cols)}  ({n_feat} признаков + label + label_binary)")
print(f"sentinel -1 сохранён: {new_neg:,}  |  raw не изменялся ✓")

processed: ../data/processed/dataset.parquet  (1248 MB)
строк: 13,691,268  |  колонок: 78  (76 признаков + label + label_binary)
sentinel -1 сохранён: 4,776,157  |  raw не изменялся ✓


In [14]:
# 10 Train / Val / Test (70/15/15) — детерминированный stratified split.
#
# Стратификация по label (14 классов). Внутри класса порядок задаётся
# хэшем вектора признаков (псевдослучайно, но воспроизводимо). Тай-брейкер
# row_id убирает неопределённость при идентичных feature-векторах (дубликатах):
# без него ROW_NUMBER на ничьих не детерминирован и сплит не воспроизводится.
#
# При ЗАПИСИ строки каждого сплита перемешиваются ORDER BY hash(row_id):
# без этого классы лежат слитными блоками (артефакт PARTITION BY label),
# и первый батч при потоковом чтении оказывается вырожденным (чистый benign),
# что ломает batch-обучение (SGD, MLP). Шаффл детерминированный.


import os
from pathlib import Path


SPLITS_DIR = Path("../data/processed/splits")
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

src = Path(CLEAN_PARQUET_PATH).as_posix()


def _q(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'


# Схема processed (без dst_port, с label_binary).
all_cols = con.execute(f"SELECT * FROM read_parquet('{src}') LIMIT 0").df().columns.tolist()
target_cols = {"label", "label_binary"}
feature_cols = [c for c in all_cols if c not in target_cols]

# Хэш по признакам (без таргетов) — псевдослучайный ключ порядка.
feature_hash = "hash(" + ", ".join(_q(c) for c in feature_cols) + ")"
# Детерминированный тай-брейкер: стабильный порядок по всем колонкам.
tiebreak = "row_number() OVER (ORDER BY " + ", ".join(_q(c) for c in all_cols) + ")"

cols_src = ", ".join(_q(c) for c in all_cols)
total = con.execute(f"SELECT COUNT(*) FROM read_parquet('{src}')").fetchone()[0]
print(f"Источник: {CLEAN_PARQUET_PATH} ({total:,} строк)\n")

print("Шаг 1/2 — вычисляем split-тег...")
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE split_tagged AS
    WITH base AS (
        SELECT {cols_src}, {tiebreak} AS row_id
        FROM read_parquet('{src}')
    ),
    numbered AS (
        SELECT {cols_src}, row_id,
               ROW_NUMBER() OVER (PARTITION BY label ORDER BY {feature_hash}, row_id) AS rn,
               COUNT(*)     OVER (PARTITION BY label)                                  AS class_n
        FROM base
    )
    SELECT {cols_src}, row_id,
           CASE
               WHEN rn <= FLOOR(class_n * 0.70) THEN 'train'
               WHEN rn <= FLOOR(class_n * 0.85) THEN 'val'
               ELSE 'test'
           END AS split_tag
    FROM numbered
""")
print("  готово ✓\n")

print("Шаг 2/2 — записываем parquet-файлы (со случайным перемешиванием строк)...")
paths = {}
for name in ['train', 'val', 'test']:
    path = (SPLITS_DIR / f"{name}.parquet").as_posix()
    paths[name] = path
    con.execute(f"""
        COPY (
            SELECT {cols_src}
            FROM split_tagged
            WHERE split_tag = '{name}'
            ORDER BY hash(row_id)        -- детерминированный шаффл внутри сплита
        )
        TO '{path}' (FORMAT PARQUET, COMPRESSION SNAPPY)
    """)
    print(f"  {name}.parquet ✓ (перемешан)")

print(f"\n  {'split':<6}  {'строк':>10}  {'%total':>7}  {'malicious':>11}  {'%mal':>6}")
print("  " + "─" * 48)
for name in ['train', 'val', 'test']:
    n   = con.execute(f"SELECT COUNT(*) FROM read_parquet('{paths[name]}')").fetchone()[0]
    n_m = con.execute(f"SELECT SUM(label_binary) FROM read_parquet('{paths[name]}')").fetchone()[0]
    print(f"  {name:<6}  {n:>10,}  {n/total*100:>6.1f}%  {int(n_m):>11,}  {n_m/n*100:>5.2f}%")

print("\nСтратификация по классам:\n")
split_class = {}
for name in ['train', 'val', 'test']:
    rows = con.execute(f"SELECT label, COUNT(*) FROM read_parquet('{paths[name]}') GROUP BY label").fetchall()
    split_class[name] = {r[0]: r[1] for r in rows}
classes = con.execute(f"SELECT label, COUNT(*) AS n FROM read_parquet('{src}') GROUP BY label ORDER BY n DESC").fetchall()
print(f"  {'label':<28}  {'total':>9}  {'train':>8}  {'val':>6}  {'test':>6}")
print("  " + "─" * 64)
for label, n_cls in classes:
    tr = split_class['train'].get(label, 0)
    va = split_class['val'].get(label, 0)
    te = split_class['test'].get(label, 0)
    print(f"  {label:<28}  {n_cls:>9,}  {tr:>8,}  {va:>6,}  {te:>6,}")

con.execute("DROP TABLE IF EXISTS split_tagged")
print(f"\nГотово → {SPLITS_DIR}/train|val|test.parquet")
print(f"Колонок: {len(all_cols)} ({len(feature_cols)} признаков + label + label_binary)")

Источник: ../data/processed/dataset.parquet (13,691,268 строк)

Шаг 1/2 — вычисляем split-тег...
  готово ✓

Шаг 2/2 — записываем parquet-файлы (со случайным перемешиванием строк)...
  train.parquet ✓ (перемешан)
  val.parquet ✓ (перемешан)
  test.parquet ✓ (перемешан)

  split        строк   %total    malicious    %mal
  ────────────────────────────────────────────────
  train    9,583,883    70.0%    2,583,883  26.96%
  val      2,053,688    15.0%      553,688  26.96%
  test     2,053,697    15.0%      553,697  26.96%

Стратификация по классам:

  label                             total     train     val    test
  ────────────────────────────────────────────────────────────────
  Benign                        10,000,000  7,000,000  1,500,000  1,500,000
  DoS Hulk                      1,802,966  1,262,076  270,445  270,445
  DDoS HOIC                     1,074,379   752,065  161,157  161,157
  DDoS LOIC-HTTP                  289,328   202,529  43,399  43,400
  FTP-Patator             

In [13]:
# Диагностика утечки дубликатов между сплитами (read-only).
# Меряем пересечение train↔val и train↔test по хэшу вектора признаков.
# Это объясняет высокие within-метрики: если значительная доля test-строк
# имеет идентичный feature-вектор в train, модель их "узнаёт", а не обобщает.

import duckdb
from pathlib import Path

try:
    con
except NameError:
    con = duckdb.connect()
    con.execute("SET memory_limit = '4GB'")
    con.execute("SET temp_directory = '/tmp/duckdb_spill'")

SPLITS = Path("../data/processed/splits")
TRAIN = (SPLITS / "train.parquet").as_posix()
VAL   = (SPLITS / "val.parquet").as_posix()
TEST  = (SPLITS / "test.parquet").as_posix()

# Признаки = всё кроме таргетов. Берём из train-схемы.
all_cols = con.execute(f"SELECT * FROM read_parquet('{TRAIN}') LIMIT 0").df().columns.tolist()
feature_cols = [c for c in all_cols if c not in ("label", "label_binary")]
fc = ", ".join('"' + c.replace('"', '""') + '"' for c in feature_cols)
print(f"Признаков для хэша: {len(feature_cols)}\n")


def overlap(name, other_path):
    """Доля строк в other_path, чей feature-вектор присутствует в train."""
    r = con.execute(f"""
        WITH tr AS (SELECT DISTINCT hash({fc}) AS h FROM read_parquet('{TRAIN}')),
             ot AS (SELECT hash({fc}) AS h, label_binary FROM read_parquet('{other_path}'))
        SELECT
            COUNT(*) AS total,
            COUNT(*) FILTER (WHERE h IN (SELECT h FROM tr)) AS in_train,
            COUNT(*) FILTER (WHERE h IN (SELECT h FROM tr) AND label_binary = 1) AS in_train_mal,
            COUNT(*) FILTER (WHERE label_binary = 1) AS total_mal
        FROM ot
    """).fetchone()
    total, in_train, in_train_mal, total_mal = r
    print(f"{name}:")
    print(f"  строк всего:                       {total:>10,}")
    print(f"  с вектором, присутствующим в train:{in_train:>10,}  ({in_train/total*100:.1f}%)")
    if total_mal > 0:
        print(f"  из них malicious:                  {in_train_mal:>10,}  "
              f"({in_train_mal/total_mal*100:.1f}% всех malicious в выборке)")
    print()


print("=" * 60)
print("ПЕРЕСЕЧЕНИЕ ПО ВЕКТОРУ ПРИЗНАКОВ (потенциальная утечка)")
print("=" * 60 + "\n")
overlap("VAL  ↔ train", VAL)
overlap("TEST ↔ train", TEST)

# Дополнительно: сколько уникальных feature-векторов вообще в train —
# показывает общий масштаб избыточности датасета.
uniq = con.execute(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT hash({fc})) AS n_uniq
    FROM read_parquet('{TRAIN}')
""").fetchone()
print("=" * 60)
print(f"Избыточность train: {uniq[0]:,} строк → {uniq[1]:,} уникальных векторов "
      f"({uniq[1]/uniq[0]*100:.1f}% уникальны, "
      f"{(1-uniq[1]/uniq[0])*100:.1f}% — повторы)")
print("=" * 60)

Признаков для хэша: 76

ПЕРЕСЕЧЕНИЕ ПО ВЕКТОРУ ПРИЗНАКОВ (потенциальная утечка)

VAL  ↔ train:
  строк всего:                        2,053,688
  с вектором, присутствующим в train:    44,385  (2.2%)
  из них malicious:                      44,378  (8.0% всех malicious в выборке)

TEST ↔ train:
  строк всего:                        2,053,697
  с вектором, присутствующим в train:    44,935  (2.2%)
  из них malicious:                      44,398  (8.0% всех malicious в выборке)

Избыточность train: 9,583,883 строк → 7,282,518 уникальных векторов (76.0% уникальны, 24.0% — повторы)
